# 05 — MCP SSE Transport

Connects to a running MCP server over **HTTP/SSE** instead of launching a subprocess.

Use the SSE transport when:
- The MCP server runs independently (e.g. in Docker)
- Multiple clients share the same server instance
- The server is remote / behind authentication

**Prerequisites**: An MCP server listening at `http://localhost:8000/sse`  
(start with e.g. `npx @modelcontextprotocol/server-everything`).

In [ ]:

from ravi.integrations.mcp import MCPClient, MCPTool
from ravi.integrations.llm.factory import create_model_client
from ravi.core.messages.client_messages import UserMessage, SystemMessage
from ravi.core.messages.content import TextBlock

from ravi.configs.settings import settings

CHAT_MODEL = settings.CHAT_MODEL
API_KEYS = {
    "openai":     settings.OPENAI_API_KEY,
    "anthropic":  settings.ANTHROPIC_API_KEY,
    "google":     settings.GEMINI_API_KEY,
    "groq":       settings.GROQ_API_KEY,
    "openrouter": settings.OPENROUTER_API_KEY,
}

## Connect via SSE transport

In [ ]:
import json
from ravi.core.messages.client_messages import ToolExecutionResultMessage

SSE_URL = "http://localhost:9000/sse"  # MCP server: docker compose -f docker/docker-compose.yml --profile mcp up -d mcp-server

async def demo_sse():
    mcp_client = MCPClient()
    try:
        await mcp_client.connect_sse(url=SSE_URL, headers={}, timeout=30.0)
        print(f"Connected via {mcp_client.transport_type} transport")

        mcp_tools = await mcp_client.discover_tools()
        print(f"Discovered {len(mcp_tools)} tools:")
        for t in mcp_tools:
            print(f"  - {t.name}: {t.description}")

        tool_map = {t.name: t for t in mcp_tools}
        client = create_model_client(CHAT_MODEL, api_keys=API_KEYS)
        messages = [
            SystemMessage(content="You are a helpful assistant with tools via MCP."),
            UserMessage(content=[TextBlock(text='Add 42 and 58, then echo back "MCP SSE works!"')]),
        ]

        response = await client.generate(
            messages=messages,
            tools=[t.get_openai_schema() for t in mcp_tools],
        )

        if response.tool_calls:
            print(f"\nLLM requested {len(response.tool_calls)} tool call(s):")
            messages.append(response)
            for tc in response.tool_calls:
                name = tc.name
                args = tc.arguments if isinstance(tc.arguments, dict) else json.loads(tc.arguments)
                print(f"  -> {name}({args})")
                tool = tool_map.get(name)
                if tool:
                    result = await tool.run(**args)
                    tool_msg = ToolExecutionResultMessage.from_tool_result(
                        tool_result=result,
                        tool_call_id=tc.tool_call_id,
                        tool_name=name,
                    )
                    print(f"  <- {tool_msg.content[0].text}")
                    messages.append(tool_msg)

            final = await client.generate(messages=messages, tools=[])
            print(f"\nConfigured chat model: {CHAT_MODEL}")
            print(f"Final response: {final.content}")
        else:
            print(f"\nDirect response: {response.content}")

    except (RuntimeError, OSError, ConnectionRefusedError) as e:
        print(f"Connection error: {e}")
        print("  Start the MCP server: docker compose -f docker/docker-compose.yml --profile mcp up -d mcp-server")
    except Exception as e:
        import traceback
        traceback.print_exc()
        print(f"Error: {type(e).__name__}: {e}")
    finally:
        if mcp_client.is_connected:
            await mcp_client.disconnect()
            print("Disconnected.")

await demo_sse()

Connected via sse transport
Discovered 7 tools:
  - add: Add two numbers.
  - subtract: Subtract b from a.
  - multiply: Multiply two numbers.
  - echo: Echo a message back.
  - to_uppercase: Convert text to uppercase.
  - word_count: Count words and characters in text.
  - server_info: Return metadata about this MCP server.

LLM requested 2 tool call(s):
  → add({'a': 42, 'b': 58})
  ← 100.0
  → echo({'message': 'MCP SSE works!'})
  ← MCP SSE works!

Final response: ['The sum of 42 and 58 is 100. MCP SSE works!']
Disconnected.


## Stdio vs SSE — side-by-side comparison

In [7]:
print('Stdio transport:')
print('  + Launches server as subprocess')
print('  + Automatic lifecycle management')
print('  + Good for local development')
print('  - One client per server instance')
print()
print('SSE transport:')
print('  + Connects to a running server')
print('  + Multiple clients can share one server')
print('  + Supports authentication headers')
print('  + Good for production / remote servers')

Stdio transport:
  + Launches server as subprocess
  + Automatic lifecycle management
  + Good for local development
  - One client per server instance

SSE transport:
  + Connects to a running server
  + Multiple clients can share one server
  + Supports authentication headers
  + Good for production / remote servers


---
## API update notes (Sprints 2 & 5)

**Sprint 2 — canonical `tool_call_id`**  
`ToolCallMessage` now exposes `.tool_call_id` (property alias for `.id`).  
All provider encoders use `.tool_call_id` — never `.id` directly.

**Sprint 5 — `discover_tools()`**  
`client.discover_tools()` replaces `MCPTool.from_mcp_client(client)`.  
Both return `list[MCPTool]`; prefer the new method.
